# 02 · Bench checks

Manual, step-by-step verification of everything added in the last iterations:
the labware resolver, definition-driven `Destination`, routine resume, cuboid
vision, floater detection, camera homography, pickup recording and the picking
session.

**How to use.** Run the one setup cell below, then run whichever check you need
— each is independent and does not rely on variables from other checks. Every
check ends with an explicit `[PASS]` / `[FAIL]` verdict; you do not have to
eyeball a DataFrame. The markdown before each check says what it verifies, what
must be on the table, and what counts as failure.

Checks are ordered by how much hardware they need:

- **Group A** — no hardware at all (run at a desk).
- **Group B** — cameras open, robot does not move.
- **Group C** — the robot moves. Disarmed by default; see the warning there.

In [ ]:
# The one shared setup. Opens no hardware; enough for Group A.
from bench_setup import *

---
## Group A — no hardware

Nothing needs to be connected. These run at a desk.

### A1 · Labware resolver

**Checks:** custom definitions in `labware/` are listed and resolve; a stock
Opentrons plate resolves from `opentrons-shared-data`; an unknown name fails with
a clear message listing what is available, not a raw traceback.
**On the table:** nothing.
**Failure:** a resolvable name errors, or an unknown name raises something other
than a clean `LabwareError`.

In [ ]:
defs = local_definitions()
print("custom definitions in labware/:", sorted(defs) or "none")
require(defs, "no custom definitions in labware/ to resolve")

custom = sorted(defs)[0]
d_local = resolve_definition(custom)
print(f"custom: {custom} -> {d_local.source}, {d_local.well_count} wells")

try:
    d_stock = resolve_definition("nest_96_wellplate_100ul_pcr_full_skirt")
    stock_note = f"stock -> v{d_stock.version} ({d_stock.source})"
except LabwareError as exc:
    stock_note = f"stock unavailable (install the 'stock' extra): {exc}"

clean_miss = False
try:
    resolve_definition("definitely_not_a_plate_xyz")
except LabwareError as exc:
    clean_miss = custom in str(exc) or "available" in str(exc).lower() or "labware/" in str(exc)
    print("unknown name ->", str(exc)[:90], "...")

ok = d_local.source == "local" and clean_miss
verdict("resolver: local, stock, and a clean miss", ok,
        expected="local resolves; unknown -> LabwareError that lists what is available",
        got=f"{custom}={d_local.source}; unknown -> {'clean LabwareError' if clean_miss else 'unexpected error'}",
        note=stock_note)

### A2 · Destination from a definition

**Checks:** the planning grid takes the plate's shape; its row/column labels come
from the definition's `ordering`; a well outside the plate is refused when the
routine is built, before any move.
**On the table:** nothing.
**Failure:** the grid labels do not match the ordering, or a bad well is accepted.

In [ ]:
name = sorted(local_definitions())[0]
dest = Destination.from_labware(name, slot=5)
table = empty_plate_table(dest)

n_rows = max(len(col) for col in dest.ordering)
n_cols = len(dest.ordering)
shape_ok = table.shape == (n_rows, n_cols)
print(f"grid {table.shape};  index {list(table.index)[:5]}...  columns {list(table.columns)[:5]}...")

# the top-left cell of the grid must map back to the first well of the ordering
probe = empty_plate_table(dest)
probe.at[table.index[0], table.columns[0]] = 1
top_left = next(iter(plan_from_table(probe, dest)))
corner_ok = top_left == dest.ordering[0][0]
print(f"cell ({table.index[0]}, {table.columns[0]}) -> well {top_left}  "
      f"(ordering starts at {dest.ordering[0][0]})")

bad_rejected = False
try:
    Routine(dest, {"ZZ999": 1}, name="check")
except RoutineError as exc:
    bad_rejected = True
    print("well outside the plate ->", str(exc)[:80])

ok = shape_ok and corner_ok and bad_rejected
verdict("Destination grid matches the definition; bad well refused", ok,
        expected="grid shaped like the plate, labels map back to ordering; bad well -> RoutineError",
        got=f"shape_ok={shape_ok} corner_ok={corner_ok} bad_well_rejected={bad_rejected}",
        note=f"plate {dest.load_name} v{dest.version}")

### A3 · Routine resume and the slot check

**Checks:** a routine written to disk, partly filled, reloads and continues from
where it stopped; on a `MockRobot`, the slot check passes for the right plate and
refuses an empty slot or a different definition.
**On the table:** nothing (uses a MockRobot).
**Failure:** the reload restarts from zero, or a wrong/empty slot is accepted.

In [ ]:
import tempfile, os as _os
name = sorted(local_definitions())[0]
dest = Destination.from_labware(name, slot=5)
wells = dest.targets[:3]

with tempfile.TemporaryDirectory() as d:
    path = _os.path.join(d, "run.json")
    r = Routine(dest, {w: 2 for w in wells}, name="bench plate", path=path)
    r.next(); r.record(delivered=1, target=wells[0])
    r.next(); r.record(delivered=2, target=wells[1])
    back = Routine.load(path)
resume_ok = (back.remaining(wells[0]) == 1 and back.remaining(wells[1]) == 0
             and back.needs_confirmation)
print("reloaded:", back.summary().splitlines()[1].strip())

robot = MockRobot()
robot.load_labware(name, 5, namespace=dest.namespace, version=dest.version)
r2 = Routine(dest, {wells[0]: 1}, name="x")

match_ok = empty_ok = wrong_ok = False
try:
    r2.check_labware(loaded_labware(robot)); match_ok = True
except RoutineError: pass
try:
    r2.check_labware({})
except RoutineError: empty_ok = True
try:
    r2.check_labware({"5": ("some_other_plate", 1)})
except RoutineError: wrong_ok = True

ok = resume_ok and match_ok and empty_ok and wrong_ok
verdict("routine resumes; slot checked (match / empty / wrong)", ok,
        expected="reload continues and needs confirmation; match passes; empty and wrong-def refuse",
        got=f"resume={resume_ok} match={match_ok} empty={empty_ok} wrong={wrong_ok}")

### A4 · Cuboid vision on a saved frame

**Checks:** the detection pipeline (detect → Otsu contour → metrics → selection)
runs on a frame from disk and marks which objects are pickable and why the rest
are not. Uses the real cuboid YOLO if its weights are in `ml_models/`, otherwise
a clearly-labelled stand-in detector so the shape/selection half still runs.
**On the table:** nothing (a synthetic dish frame ships in `tests/fixtures/bench/`).
**Failure:** the pipeline errors, or the object count is wildly off.

In [ ]:
data = ensure_bench_data()
frame = cv2.imread(str(data["frame"]))
require(frame is not None, f"could not read {data['frame']}")
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
cx, cy, rad = data["dish"]
one_d = 0.0125                       # mm/px for this synthetic frame
cfg = PickingConfig(circle_center=(cx, cy), circle_radius=rad,
                    cuboid_size_threshold=(250, 500), minimum_distance=1.7)

weights = paths.ml_models_dir() / "cuboid_bbox_v4-11_best.pt"
if weights.exists():
    from ultralytics import YOLO
    boxes, confs = vision.detect_boxes(YOLO(str(weights)), frame, cfg)
    detector_note = "real cuboid YOLO"
else:
    boxes, confs = blob_boxes(gray)
    detector_note = "stand-in blob detector (real YOLO weights not in ml_models/)"

roi = vision.roi_mask(gray, cfg, one_d)
df = vision.build_cuboid_df(gray, boxes, confs, roi_mask=roi,
                            pad=cfg.otsu_pad, open_k=cfg.otsu_open_k)
df = vision.add_derived(df, one_d * one_d, one_d, cfg.circle_center)
if len(df):
    keep = vision.select_pickable(df, cfg)
    pickable = df.loc[keep]
    isolated = pickable.loc[pickable.min_dist_mm > cfg.minimum_distance]
else:
    pickable = isolated = df

show(overlays.annotate(frame, cuboid_df=df, pickable=pickable, isolated=isolated,
                       circle_center=cfg.circle_center, circle_radius=cfg.circle_radius,
                       status_lines=[f"detected {len(df)}", f"pickable {len(pickable)}",
                                     f"isolated {len(isolated)}"]),
     "red = all detections, yellow = pickable, green = isolated")

if len(df):
    view = df[["cX", "cY", "diameter_microns", "circularity", "min_dist_mm",
               "distance_to_center"]].round(1).copy()
    view["pickable"] = keep.values
    print(view.to_string(index=False))

expected = len(data["blobs"])
ok = len(df) == expected
verdict("cuboid vision: detect, metrics, selection", ok,
        expected=f"{expected} objects detected",
        got=f"{len(df)} detected, {len(pickable)} pickable, {len(isolated)} isolated",
        note=detector_note + "; oversized fails size, the close pair fails spacing")

### A5 · Floater detection on a saved clip

**Checks:** temporal-variance floater detection runs on a short clip from disk,
separately from the picking loop, and finds the moving region.
**On the table:** nothing (a synthetic clip ships in `tests/fixtures/bench/`).
**Failure:** no zone is found where the clip clearly has motion, or it errors.

In [ ]:
clip = load_bench_clip()
vmap = vision.temporal_variance(clip, downscale=2, step=1)
show(heatmap(vmap), "temporal variance (bright = moved most)")
zones = vision.detect_floater_zones(clip, downscale=2, min_area=3, mad_k=5.0)
print("zones (video px):", [(round(x), round(y)) for x, y in zones])

near = any(abs(x - 200) < 25 and abs(y - 150) < 25 for x, y in zones)
ok = len(zones) == 1 and near
verdict("floater detection on a saved clip", ok,
        expected="one zone near (200, 150)",
        got=f"{len(zones)} zone(s): {[(round(x), round(y)) for x, y in zones]}")

---
## Group B — cameras open, robot still

These open the camera(s) they need and close them again. The robot is read from
but never commanded to move. Each check is independent: it loads the profile and
opens its own cameras.

### B6 · Both cameras

**Checks:** the overview and underview cameras open, each returns a frame with
`ret` handled, and the resolution matches the profile's `CameraSpec`. There is no
separate undistort stage — frames are used raw by design (DESIGN §3) — so this
also confirms the pixel map, if present, matches the camera.
**On the table:** both cameras connected; anything in view.
**Failure:** a camera does not open, returns no frame, or reports an unexpected
resolution.

In [ ]:
profile = load_profile()
cams = open_cameras(profile)
results = {}
try:
    for label in ("overview_cam", "underview_cam"):
        require(label in profile.cameras, f"profile has no camera {label!r}")
        cam = cams.open(label)
        ret, frame = cam.read()
        if not ret or frame is None:            # a fresh camera may need a moment
            frame = cam.read_after(time.monotonic())
            ret = frame is not None
        want = tuple(profile.cameras[label].default_resolution)
        res_ok = tuple(cam.resolution) == want
        results[label] = dict(ret=ret, res=tuple(cam.resolution), want=want, res_ok=res_ok)
        if ret:
            small = cv2.resize(frame, (480, int(480 * frame.shape[0] / frame.shape[1])))
            show(small, f"{label}")
finally:
    cams.close_all()

map_note = "no pixel map in profile"
if profile.calibration.pixel_map is not None and results.get("overview_cam", {}).get("ret"):
    problems = profile.pixel_map.check_camera(results["overview_cam"]["res"])
    map_note = "pixel map matches the overview camera" if not problems else "; ".join(problems)

ok = all(r["ret"] and r["res_ok"] for r in results.values())
verdict("both cameras: frame returned, ret handled, resolution matches", ok,
        expected="each camera returns a frame at its profile resolution",
        got=results, note="frames used raw, no undistort stage; " + map_note)

### B7 · Live floater detection (overview)

**Checks:** floater detection on a short live burst from the overview camera.
**On the table:** the dish under the overview camera. A still dish gives few or no
zones; nudge a floating piece to see a zone appear.
**Failure:** the camera delivers too few frames, or the pipeline errors.

In [ ]:
profile = load_profile()
cams = open_cameras(profile)
frames = []
try:
    over = cams.open("overview_cam")
    end = time.monotonic() + 2.0
    while time.monotonic() < end:
        ret, fr = over.read()
        if ret and fr is not None:
            frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
finally:
    cams.close_all()

require(len(frames) >= 2, f"only {len(frames)} frames grabbed; the camera is not delivering")
vmap = vision.temporal_variance(frames, downscale=4, step=2)
show(heatmap(vmap), "live temporal variance (overview)")
zones = vision.detect_floater_zones(frames, downscale=4,
                                    min_area=profile.picking.floater_min_area,
                                    mad_k=profile.picking.floater_mad_k)
print("zones (video px):", [(round(x), round(y)) for x, y in zones])
verdict("live floater detection ran", True,
        expected="a variance map and a (possibly empty) list of zones",
        got=f"{len(frames)} frames over ~2 s, {len(zones)} zone(s)",
        note="still dish -> few/no zones; wave a floater to make one appear")

### B8 · Camera homography (standalone)

**Checks:** the upper→lower homography from the crosshair disc in both cameras,
with the reprojection error, and a round-trip sanity check: the disc centre
mapped over→under and back must land on itself.
**On the table:** the crosshair calibration disc under the overview camera, in
view of both cameras. No pipette move happens.
**Failure:** the reprojection error is large, or the round-trip does not return to
the centre.

In [ ]:
profile = load_profile()
require(profile.calibration.pixel_map is not None,
        "no pixel map; run camera calibration (notebook 01, section 3) first")
weights = paths.ml_models_dir() / profile.calibration.tip_target.model_file
require(weights.exists(), f"tip model {weights} missing (needed to find the crosshairs)")

robot = connect_robot()                       # reads the gantry pose; no move
cams = open_cameras(profile)
try:
    over = cams.open("overview_cam"); under = cams.open("underview_cam")
    from ultralytics import YOLO
    tip = TipDetector(YOLO(str(weights)),
                      imgsz=profile.calibration.tip_target.imgsz,
                      conf=profile.calibration.tip_target.conf)
    res = calibrate_homography(robot, over, under, tip,
                               target=profile.calibration.tip_target)
    print(res.report)

    H = np.asarray(res.homography.matrix, dtype=np.float64)
    centre = np.asarray(res.over_view.centre, dtype=np.float32).reshape(1, 1, 2)
    under_px = cv2.perspectiveTransform(centre, H)
    back = cv2.perspectiveTransform(under_px, np.linalg.inv(H)).reshape(2)
    roundtrip_px = float(np.linalg.norm(back - np.asarray(res.over_view.centre)))
finally:
    cams.close_all()

ok = res.report.reproj_max_px < 3.0 and roundtrip_px < 1.0
verdict("homography: reprojection and centre round-trip", ok,
        expected="reproj max < 3 px and the centre returns to itself < 1 px",
        got=f"reproj mean {res.report.reproj_mean_px:.2f} max {res.report.reproj_max_px:.2f} px; "
            f"round-trip {roundtrip_px:.2f} px")

### B9 · Recorder

**Checks:** a few seconds recorded from the underview camera, with a ROI box, save
to a file, and the file appears. Plus: with recording off, the session creates no
recorder at all.
**On the table:** the underview camera connected; anything in view.
**Failure:** no file is written, or a session built with `clip_dir=None` still
holds a recorder.

In [ ]:
profile = load_profile()
cams = open_cameras(profile)
clip_path = paths.clips_dir() / f"bench_recorder_{time.strftime('%H%M%S')}.mp4"
file_ok = False
try:
    under = cams.open("underview_cam")
    rec = under.record(max_frames=profile.picking.clip_max_frames)
    w, h = under.resolution
    rec.mark_roi(w // 2, h // 2, half=80)
    rec.start(); time.sleep(3.0); rec.stop()
    rec.save(str(clip_path), color=True)
    under.detach(rec)
    file_ok = clip_path.exists() and clip_path.stat().st_size > 0
    print("wrote", clip_path, f"({clip_path.stat().st_size} bytes)" if file_ok else "(missing)")
finally:
    cams.close_all()

# recording off: no recorder is created (checked on mocks, no hardware)
off_ok = mock_session(clip_dir=None)._recorder is None

ok = file_ok and off_ok
verdict("recorder writes a clip; off means no recorder", ok,
        expected="a non-empty file on disk; clip_dir=None -> no recorder",
        got=f"file_written={file_ok} recorder_off_is_none={off_ok}")

---
## ⚠️ Group C — the robot MOVES

The checks below command the real robot. **They are disarmed by default.** A
stray *Run All* in a notebook that drives a robot is expensive, so every check
here refuses to run until you set the flag in the next cell to `True` by hand,
and each check re-reads that flag, so running one alone is still gated.

Make sure the deck is clear and you are watching the robot before you arm this.

In [ ]:
# Flip to True by hand to allow the Group C checks to move the robot.
ROBOT_MOVES_ARMED = False
print("Group C is", "ARMED — the robot will move" if ROBOT_MOVES_ARMED else "disarmed")

### C10 · Loaded labware vs the destination slot

**Checks:** what is actually loaded in each slot of the current run, and whether
the destination slot from the config holds a plate. Read-only, but gated with the
rest of Group C.
**On the table:** the plate loaded into its slot, the run created.
**Failure:** the destination slot is empty or holds something unexpected.

In [ ]:
require(globals().get("ROBOT_MOVES_ARMED", False),
        "Group C is disarmed — set ROBOT_MOVES_ARMED = True to run it")
profile = load_profile()
robot = connect_robot()
loaded = loaded_labware(robot)
for slot in sorted(loaded, key=lambda s: int(s)):
    lw = loaded[slot]
    print(f"  slot {slot}: {lw.load_name} v{lw.version} ({lw.namespace})")

slot = str(profile.picking.destination_slot)
here = loaded.get(slot)
ok = here is not None
verdict(f"destination slot {slot} holds a plate", ok,
        expected=f"a plate in slot {slot}",
        got=(f"{here.load_name} v{here.version}" if here else "empty"),
        note="compare this against the plate your routine expects")

### C11 · One `step()` of the picking session

**Checks:** a single transition of the picking session, printing the event, the
new state, the objects detected and the current well. No loop.
**On the table:** a calibrated profile, the cuboid model in `ml_models/`, the dish
under the overview camera, the destination plate loaded.
**Failure:** `step()` errors, or a prerequisite is missing without a clear message.

In [ ]:
require(globals().get("ROBOT_MOVES_ARMED", False),
        "Group C is disarmed — set ROBOT_MOVES_ARMED = True to run it")
profile = load_profile()
require(profile.calibration.is_ready, "profile is not calibrated; run 01 sections 3-4 first")
weights = paths.ml_models_dir() / "cuboid_bbox_v4-11_best.pt"
require(weights.exists(), f"cuboid model {weights} missing")

pmap = PixelMap.from_config(profile.pixel_map)
cams = open_cameras(profile)
robot = connect_robot()
try:
    over = cams.open("overview_cam")
    from ultralytics import YOLO
    model = YOLO(str(weights))
    loaded = loaded_labware(robot)
    slot = str(profile.picking.destination_slot)
    require(slot in loaded, f"no plate in slot {slot}")
    dest = Destination.from_labware(loaded[slot].load_name, profile.picking.destination_slot,
                                    version=loaded[slot].version)
    routine = Routine(dest, {dest.targets[0]: 1}, name="bench step")
    session = PickingSession(robot, over, pmap, profile, routine, model,
                             labware_id=robot.labware_dct[slot])
    event = session.step()
    print(event)
    print("state:", session.state.value, "| detected:", len(session.cuboid_df),
          "| current well:", routine.current)
    verdict("one step() ran and returned an event", True,
            expected="a PickEvent and a new state, no exception",
            got=str(event))
finally:
    cams.close_all()

### C12 · pause / stop lands between moves

**Checks:** the stop flag takes effect *between individual moves*, not only
between states. A small gated loop of ±1 mm moves is stopped from another thread
after the first move; it must halt before running them all.
**On the table:** clear deck; the moves are tiny but real.
**Failure:** all the moves run despite stop being set early.

In [ ]:
require(globals().get("ROBOT_MOVES_ARMED", False),
        "Group C is disarmed — set ROBOT_MOVES_ARMED = True to run it")
import threading
from micropick.hardware.protocols import move_relative

robot = connect_robot()
start = np.array(xyz(robot))
steps = [("x", 1.0), ("x", -1.0), ("y", 1.0), ("y", -1.0), ("x", 1.0), ("x", -1.0)]
stop = threading.Event()
state = {"done": 0}

def watcher():
    while state["done"] < 1:
        time.sleep(0.005)
    stop.set()                                  # stop right after the first move

t = threading.Thread(target=watcher); t.start()
try:
    for axis, d in steps:
        if stop.is_set():                       # the gate, between moves
            break
        move_relative(robot, axis, d)
        state["done"] += 1
finally:
    t.join()
    goto_xy(robot, float(start[0]), float(start[1]))     # return to where we began

ok = 0 < state["done"] < len(steps)
verdict("stop lands between moves, not only between states", ok,
        expected=f"stopped before all {len(steps)} moves ran",
        got=f"{state['done']} of {len(steps)} moves executed")

### C13 · `finally` raises Z after an interruption

**Checks:** an interrupted sequence still raises the Z axis in its `finally`. The
tip is lowered a little, an exception is raised mid-sequence, and the `finally`
retracts Z; Z must end higher than at the bottom. (You can also interrupt the
cell by hand — the `finally` runs either way.)
**On the table:** clear deck below the tip.
**Failure:** Z stays down after the interruption.

In [ ]:
require(globals().get("ROBOT_MOVES_ARMED", False),
        "Group C is disarmed — set ROBOT_MOVES_ARMED = True to run it")
from micropick.hardware.protocols import move_relative

robot = connect_robot()
z0 = xyz(robot)[2]
z_bottom = None
try:
    move_relative(robot, "z", -5.0)             # lower the tip a little
    z_bottom = xyz(robot)[2]
    raise KeyboardInterrupt("simulated interruption mid-sequence")
except KeyboardInterrupt as exc:
    print("interrupted:", exc)
finally:
    robot.retract_axis("leftZ")                 # the finally that must raise Z

z_after = xyz(robot)[2]
ok = z_bottom is not None and z_after > z_bottom + 0.5
verdict("finally raised Z despite the interruption", ok,
        expected="Z ends higher than at the bottom",
        got=f"start={z0:.1f}  bottom={z_bottom:.1f}  after={z_after:.1f} mm")